# Protein Applications

Load an ordered backbone trace with the public PDB and mmCIF adapters, then
inspect atom/chain selection, the source record and the resulting embedding.
This notebook uses a local PDB example that runs offline.

For contact, residue, cavity or domain graphs, define the nodes and edges for
your scientific question before following the shared topology workflow.


> **Notebook setup**
>
> - **Audience:** first-time users loading biomolecular backbone data.
> - **Environment:** base 0.2 development API.
> - **Example:** a temporary local PDB file, loaded offline.
> - **Result:** an ordered backbone curve and text diagnostics.


## 1. What you will inspect

The example makes four checks that also apply to a real local file or an RCSB
download: atom/chain selection, source records, returned issues, and the common
embedded-graph `pos`/`pts` contract.


In [ ]:
from pathlib import Path
import tempfile

import knotted_graph
from knotted_graph.inputs import from_protein_ca_backbone

print("KnottedGraph version:", knotted_graph.__version__)


## 2. Load a local C-alpha backbone

A real PDB may contain several models, chains, atom names, and alternate
locations. This example file contains one model and one chain. For a multi-chain
structure, select the chain explicitly with `chain_id`.


In [ ]:
def pdb_atom(serial, residue, sequence, x, y, z):
    return (
        f"ATOM  {serial:5d} {'CA':>4s} {residue:>3s} A{sequence:4d}    "
        f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00 20.00           C\n"
    )

temporary_directory = tempfile.TemporaryDirectory()
pdb_path = Path(temporary_directory.name) / "backbone_demo.pdb"
pdb_path.write_text(
    "".join(
        [
            pdb_atom(1, "ALA", 1, 0.0, 0.0, 0.0),
            pdb_atom(2, "GLY", 2, 1.2, 0.1, 0.0),
            pdb_atom(3, "SER", 3, 1.8, 1.0, 0.4),
            pdb_atom(4, "VAL", 4, 2.6, 1.4, 1.1),
        ]
    ),
    encoding="utf-8",
)

result = from_protein_ca_backbone(
    pdb_path,
    pdb_id="DEMO",
    chain_id="A",
    model_id=1,
)

print("selected chain:", result.chain_id)
print("available chains:", dict(result.available_chains))
print("coordinate shape:", result.coords.shape)
print("residues:", [record["residue_name"] for record in result.records])
print("issues:", result.issues)


## 3. Inspect the returned graph

The ordered samples are stored as the geometry of one open graph edge. They are
not converted into four topological graph vertices. This distinction matters:
coordinate sampling density should not change the abstract graph topology.


In [ ]:
graph = result.graph
u, v, key = next(iter(graph.edges(keys=True)))
points = graph.edges[u, v, key]["pts"]

print("graph nodes / edges:", graph.number_of_nodes(), graph.number_of_edges())
print("edge sample count:", len(points))
print("start agrees with node pos:", (points[0] == graph.nodes[u]["pos"]).all())
print("end agrees with node pos:", (points[-1] == graph.nodes[v]["pos"]).all())
print("input kind:", graph.graph["input_kind"])


## 4. Decide what your scientific graph means

If the ordered backbone is your intended curve, continue with embedding
validation and the core workflow. If you instead need a contact, residue,
cavity, or domain graph, first define: (1) what a node represents; (2) the rule
that creates an edge; (3) coordinate units and closure; and (4) perturbation or
validation checks. Use these definitions to construct the graph that represents
your scientific model.

To relax the embedding before projection, use the optional native Repulsor
workflow described in the Repulsive Layout guide.


## 5. Continue

Read the web **Input handling** page for PDB/mmCIF atom-selection, download,
closure, and parser details. Then use **02 - Core Workflows** for graph
validation, projection, and invariant provenance. Choose closure according to
your scientific model and record it with the graph construction. Interpret a
zero polynomial in the conventions appropriate to the chosen curve and its
projection.
